# R3d — Semantic-Diff Quantum Rewrite

Creates **semantically DIFFERENT** variants of quantum tasks (MRI paper §3.1 style).

Pipeline per task:
1. **Rewrite Agent** (DeepSeek-R1) — change one gate or gate order → new code + new prompt + new test  
2. **Self-validation** — ② new_test must PASS on SD canonical; ③ orig_test must FAIL on SD canonical  
3. **Judge Agent** (DeepSeek-R1) — alignment + difficulty check → ACCEPT / REJECT  
4. **Store** — all records (pass + reject) written to `semantic_diff_results.json`

Strategies: `diff_gate` | `diff_sequence`

## 1. Setup

In [60]:
import sys, json, re, time, random
from pathlib import Path

sys.modules.pop('qmri_utils', None)
import qmri_utils as _qu
from qmri_utils import execute_code

LABELING_PATH = 'exp1_results/strategy_labeling.json'
EXC_ARR_PATH  = 'exp1_results/exc_arr.json'
OUT_PATH      = 'exp1_results/qmri_rewrite_diff_results.json'

with open(LABELING_PATH, encoding='utf-8') as f:
    LABELING = json.load(f)
with open(EXC_ARR_PATH, encoding='utf-8') as f:
    EXC_LOOKUP = {r['task_id']: r for r in json.load(f)}

print(f'labeling: {len(LABELING)} tasks  |  exc_arr: {len(EXC_LOOKUP)} records')

labeling: 195 tasks  |  exc_arr: 195 records


In [61]:
import subprocess, sys
result = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pylatexenc'], capture_output=True, text=True)
print(result.stdout or 'already up to date')
if result.returncode != 0:
    print('INSTALL FAILED:', result.stderr)

already up to date


## 2. API Client & Prompts

In [62]:
import re as _re, time as _time, random as _random
import openai as _openai
from openai import OpenAI as _OpenAI
from openai import AzureOpenAI as _AzureOpenAI

# ── DeepSeek-R1（rewrite） ────────────────────────────────────────────────────
CONFIG = {
    'ds_deployment':  'DeepSeek-R1-39794c',
    'ds_endpoint':    'https://moonds39794c.services.ai.azure.com/models',
    'ds_api_key':     'hello',
    'ds_api_version': '2024-05-01-preview',
    'temperature':    1.0,
    'max_tokens':     16000,
}

# ── GPT 5.4（judge） ──────────────────────────────────────────────────────────
GPT54_JUDGE_CONFIG = {
    'deployment':  'gpt-5.4-dbe387',
    'endpoint':    'https://moon-gpt54dbe387.cognitiveservices.azure.com/',
    'api_key':     'hello',
    'api_version': '2025-04-01-preview',
}

STRATEGY_DESC = {
    'diff_gate': (
        'Replace exactly ONE standard named gate with a DIFFERENT standard named gate '
        'that changes the circuit semantics.\n'
        'Valid gate replacements: CX→CZ, H→X, S→T, CX→CY, X→Y, T→S, CZ→CX, etc.\n'
        'INVALID replacements:\n'
        '  - Replacing a gate with an equivalent decomposition (e.g. CX→H·CZ·H)\n'
        '  - Replacing parametric calls like unitary(), initialize(), or prepare_state() '
        '— these define the whole task structure and must not be touched\n'
        'The replacement must produce a circuit with a different quantum state or measurement distribution.'
    ),
    'diff_sequence': (
        'Swap the positions of exactly TWO gates that act on overlapping qubits (non-commuting) '
        'to change the circuit semantics.\n'
        'The two gates must share at least one qubit so their order matters.\n'
        'The two gates must be mathematically non-commuting ([A,B] ≠ 0) — '
        'verify that swapping them yields a genuinely different quantum state.\n'
        'After the swap, the gate sequence in qc.data must visibly differ '
        '(the instruction order in qc.data changes).\n'
        'INVALID: swapping gates on completely disjoint qubits (those always commute).\n'
        'INVALID: swapping two identical gates (A=B → [A,B]=0, trivially commute).\n'
        'INVALID: changing any gate names — only reorder, do not add/remove/rename gates.\n'
        'The swap must produce a circuit with a different quantum state or measurement distribution.'
    ),
}

REWRITE_SYS = (
    'You are a quantum computing expert specialising in Qiskit >= 1.3.0. '
    'Your task is to create a SEMANTICALLY DIFFERENT variant of a quantum circuit task '
    'by modifying the canonical solution. Follow the output format exactly.'
)

REWRITE_TMPL = """\
Given this quantum computing task and its canonical solution:

Canonical Solution (complete, runnable):
```python
{full_canonical}
```

Original Test:
```python
{orig_test}
```

=== YOUR TASK ===
Strategy: {strategy_name}
{strategy_desc}

Rules:
- Keep the SAME function signature (name, parameters, return type).
- The modified circuit must produce a DIFFERENT result (different state or measurement distribution).
- The New Prompt must explicitly describe the NEW behaviour — do NOT copy the original prompt.
- Write a New Test in the SAME FORMAT as the Original Test, but verifying the NEW behaviour.
  (If Original Test uses `def check(candidate)`, New Test must also use `def check(candidate)`.)
- Include all necessary Qiskit imports at the top of the New Code block.
- Target Qiskit >= 1.3.0; no deprecated APIs.
- Do NOT mention this is a modification or reference the original solution.

Output in EXACTLY this format (no text outside these sections):

New Code:
```python
[complete function, including all needed imports]
```
Gate Change:
[one line: exactly what was changed, e.g. "replaced cx(0,1) with cz(0,1)" or "swapped h(0) and cx(0,1)"]
New Prompt:
[updated task description reflecting the new semantics]
New Test:
```python
[test in same format as Original Test, verifying NEW behaviour]
```"""

JUDGE_SYS = (
    'You are an expert quantum computing code reviewer. '
    'Evaluate whether a semantically-different rewrite meets quality standards.'
)

JUDGE_TMPL = """\
Evaluate this quantum circuit task rewrite:

**Original Task (canonical solution):**
```python
{full_canonical}
```

**Rewritten Task:**
Prompt: {new_prompt}
Code:
```python
{new_code}
```

Gate Change: {gate_change}

**Evaluation Criteria:**
1. Prompt-Code Alignment: Does the new prompt accurately describe what the new code does?
2. Difficulty Equivalence: Is the rewritten task of similar difficulty (same algorithmic complexity, same Qiskit knowledge required)?
3. Semantic Difference: Does the gate change genuinely alter the circuit output (not just a cosmetic rename or equivalent decomposition)?

**Response Format:**
Overall Recommendation: [ACCEPT or REJECT]
Reason: [one or two sentences explaining the decision, especially if rejecting]"""

_ds_client    = None
_gpt54_client = None


def _call_ds(system: str, user: str, max_retries: int = 2) -> str:
    """DeepSeek-R1 rewrite call with Retry-After support."""
    global _ds_client
    if _ds_client is None:
        _ds_client = _OpenAI(
            base_url=CONFIG['ds_endpoint'],
            api_key=CONFIG['ds_api_key'],
            default_query={'api-version': CONFIG['ds_api_version']},
        )
    messages = [{'role': 'system', 'content': system},
                {'role': 'user',   'content': user}]
    for attempt in range(max_retries):
        try:
            resp = _ds_client.chat.completions.create(
                model=CONFIG['ds_deployment'],
                messages=messages,
                temperature=CONFIG['temperature'],
                max_tokens=CONFIG['max_tokens'],
            )
            content = resp.choices[0].message.content or ''
            return _re.sub(r'<think>.*?</think>', '', content, flags=_re.DOTALL).strip()
        except _openai.RateLimitError as e:
            retry_after = None
            try:
                retry_after = int(e.response.headers.get('Retry-After', 0)) or None
            except Exception:
                pass
            if retry_after:
                delay = retry_after + _random.uniform(1, 5)
                label = f'Retry-After={retry_after}s'
            else:
                delay = min(120, 2 ** attempt) + _random.uniform(0, 2)
                label = 'exp-backoff'
            print(f'  [DS 429] attempt {attempt+1}/{max_retries} [{label}] — sleep {delay:.1f}s')
            if attempt < max_retries - 1:
                _time.sleep(delay)
            else:
                raise
        except Exception:
            raise
    raise RuntimeError(f'LLM failed after {max_retries} attempts')


def _call_judge(system: str, user: str, max_retries: int = 6) -> str:
    """GPT 5.4 judge call with Retry-After support."""
    global _gpt54_client
    if _gpt54_client is None:
        _gpt54_client = _AzureOpenAI(
            api_key=GPT54_JUDGE_CONFIG['api_key'],
            api_version=GPT54_JUDGE_CONFIG['api_version'],
            azure_endpoint=GPT54_JUDGE_CONFIG['endpoint'],
        )
    messages = [{'role': 'system', 'content': system},
                {'role': 'user',   'content': user}]
    for attempt in range(max_retries):
        try:
            resp = _gpt54_client.chat.completions.create(
                model=GPT54_JUDGE_CONFIG['deployment'],
                messages=messages,
                temperature=0.0,
                max_completion_tokens=1024,
            )
            return resp.choices[0].message.content or ''
        except _openai.RateLimitError as e:
            retry_after = None
            try:
                retry_after = int(e.response.headers.get('Retry-After', 0)) or None
            except Exception:
                pass
            if retry_after:
                delay = retry_after + _random.uniform(1, 5)
                label = f'Retry-After={retry_after}s'
            else:
                delay = min(120, 2 ** attempt) + _random.uniform(0, 2)
                label = 'exp-backoff'
            print(f'  [judge 429] attempt {attempt+1}/{max_retries} [{label}] — sleep {delay:.1f}s')
            if attempt < max_retries - 1:
                _time.sleep(delay)
            else:
                raise
        except Exception:
            raise
    raise RuntimeError(f'judge LLM failed after {max_retries} attempts')


def build_rewrite_msg(task: dict, strategy: str) -> str:
    full_canonical = build_full_code(
        task.get('original_prompt_text', ''),
        task.get('canonical_solution_text', '')
    )
    return (REWRITE_TMPL
        .replace('{full_canonical}', full_canonical)
        .replace('{orig_test}',      task.get('test', ''))
        .replace('{strategy_name}',  strategy)
        .replace('{strategy_desc}',  STRATEGY_DESC[strategy])
    )

def build_judge_msg(task: dict, parsed: dict) -> str:
    full_canonical = build_full_code(
        task.get('original_prompt_text', ''),
        task.get('canonical_solution_text', '')
    )
    return (JUDGE_TMPL
        .replace('{full_canonical}', full_canonical)
        .replace('{new_prompt}',     parsed.get('new_prompt', ''))
        .replace('{new_code}',       parsed.get('new_code', ''))
        .replace('{gate_change}',    parsed.get('gate_change', ''))
    )

print('API client + prompts ready.')
print('  rewrite → DeepSeek-R1 (max_retries=2)  |  judge → GPT 5.4 (max_retries=6)')

API client + prompts ready.
  rewrite → DeepSeek-R1 (max_retries=2)  |  judge → GPT 5.4 (max_retries=6)


## 3. Response Parser & Validation

In [63]:
import contextlib, io as _io

def _extract_block(text, tag):
    # Tolerate optional markdown bold (**Tag:**) around the header
    pattern = rf'(?i)\*{{0,2}}{re.escape(tag)}\*{{0,2}}:\s*'
    parts = re.split(pattern, text, maxsplit=1)
    if len(parts) < 2:
        return ''
    after = parts[1]
    next_sec = re.search(
        r'(?i)^(?:\*{0,2})(new code|gate change|new prompt|new test|overall recommendation|reason)(?:\*{0,2}):\s*',
        after, re.MULTILINE
    )
    return after[:next_sec.start()].strip() if next_sec else after.strip()

def _extract_code(text):
    m = re.search(r'```python\s*([\s\S]*?)```', text)
    if m:
        return m.group(1).strip()
    m = re.search(r'```\s*([\s\S]*?)```', text)
    if m:
        return m.group(1).strip()
    return text.strip()

def parse_rewrite(raw):
    new_code_blk = _extract_block(raw, 'New Code')
    new_test_blk = _extract_block(raw, 'New Test')

    # Fallback: scan all ```python blocks
    all_blocks = re.findall(r'```python\s*([\s\S]*?)```', raw)

    # Test fallback: last block
    if not new_test_blk and len(all_blocks) >= 2:
        new_test_blk = all_blocks[-1]
    # Code fallback: first block (only if test took a different block)
    if not new_code_blk and len(all_blocks) >= 1:
        new_code_blk = all_blocks[0]

    return {
        'new_code':    _extract_code(new_code_blk),
        'new_test':    _extract_code(new_test_blk),
        'gate_change': _extract_block(raw, 'Gate Change').split('\n')[0].strip(),
        'new_prompt':  _extract_block(raw, 'New Prompt'),
    }

def parse_judge(raw):
    rec_m    = re.search(r'(?i)overall recommendation:\s*(ACCEPT|REJECT)', raw)
    reason_m = re.search(r'(?i)reason:\s*(.+)', raw, re.DOTALL)
    return {
        'recommendation': rec_m.group(1).upper() if rec_m else 'UNKNOWN',
        'reason':         reason_m.group(1).strip()[:500] if reason_m else raw[:300],
    }

def build_full_code(prompt_text, solution_text):
    code = prompt_text.rstrip() + '\n' + solution_text
    try:
        compile(code, '<string>', 'exec')
        return code
    except SyntaxError:
        return prompt_text.rstrip() + '\n    ' + solution_text

# ── Gate structure helpers ────────────────────────────────────────────────────

STANDARD_GATES = {
    'h', 'x', 'y', 'z', 's', 'sdg', 't', 'tdg', 'sx', 'sxdg', 'id',
    'rx', 'ry', 'rz', 'p', 'u', 'u1', 'u2', 'u3',
    'cx', 'cy', 'cz', 'ch', 'cp', 'crx', 'cry', 'crz', 'cu', 'cu1', 'cu3',
    'swap', 'iswap', 'dcx', 'ecr',
    'ccx', 'cswap', 'ccz',
    'r', 'rv', 'ms',
}

def _get_gate_ops(code, entry_point, kwargs=None):
    """Run code and return [(gate_name, qubit_indices)], or None on error."""
    ns = {}
    try:
        with contextlib.redirect_stdout(_io.StringIO()), contextlib.redirect_stderr(_io.StringIO()):
            exec(code, ns)
        fn = ns[entry_point]
        try:
            result = fn(**(kwargs or {}))
        except TypeError:
            result = fn()
        if isinstance(result, tuple):
            result = result[0]
        qc = result.copy()
        qc.remove_final_measurements(inplace=True)
        return [
            (instr.operation.name, tuple(qc.find_bit(q).index for q in instr.qubits))
            for instr in qc.data
            if instr.operation.name not in ('barrier', 'delay')
        ]
    except Exception:
        return None

def check_gate_structure(orig_full, sd_full, entry_point, strategy, kwargs=None):
    """
    diff_gate:
      - Same total gate count.
      - All changed gates share ONE orig type → ONE sd type (handles loops).
      - Both types must be standard named gates (not unitary/initialize etc.).
    diff_sequence:
      - Same gate count, same gate multiset, order must differ.
    Returns (ok: bool, message: str).
    """
    orig_ops = _get_gate_ops(orig_full, entry_point, kwargs)
    sd_ops   = _get_gate_ops(sd_full,   entry_point, kwargs)

    if orig_ops is None:
        return False, 'struct_check: could not run original circuit'
    if sd_ops is None:
        return False, 'struct_check: could not run rewritten circuit'
    if len(orig_ops) != len(sd_ops):
        return False, f'struct_check: gate count changed {len(orig_ops)} → {len(sd_ops)}'

    if strategy == 'diff_gate':
        diffs = [(o, s) for o, s in zip(orig_ops, sd_ops) if o[0] != s[0]]
        if len(diffs) == 0:
            return False, 'struct_check: no gate names changed'

        # Allow loop replacement: all diffs share ONE orig type and ONE sd type
        orig_types = {o[0] for o, s in diffs}
        sd_types   = {s[0] for o, s in diffs}
        if len(orig_types) > 1 or len(sd_types) > 1:
            return False, (f'struct_check: {len(diffs)} gates differ with '
                           f'{len(orig_types)} original types — not a single gate change')

        orig_name = next(iter(orig_types))
        sd_name   = next(iter(sd_types))
        if orig_name not in STANDARD_GATES:
            return False, (f'struct_check: original gate {orig_name!r} is not a '
                           f'standard named gate (cannot apply diff_gate)')
        if sd_name not in STANDARD_GATES:
            return False, f'struct_check: rewritten gate {sd_name!r} is not a standard named gate'

        n = len(diffs)
        label = f'{orig_name} → {sd_name}' + (f' ×{n}' if n > 1 else f' on qubits {diffs[0][0][1]}')
        return True, label

    elif strategy == 'diff_sequence':
        orig_names = sorted(o[0] for o in orig_ops)
        sd_names   = sorted(s[0] for s in sd_ops)
        if orig_names != sd_names:
            return False, 'struct_check: gate names changed (only order should change)'
        if orig_ops == sd_ops:
            return False, 'struct_check: gate sequence unchanged'
        return True, 'gate order changed'

    return True, 'no structural check for this strategy'

# ── Validation ────────────────────────────────────────────────────────────────

def validate_sd(sd_full, new_test, orig_test, entry_point, orig_full, timeout=30.0):
    r2 = execute_code(sd_full, new_test,  entry_point, canonical_solution=sd_full,   timeout=timeout)
    r3 = execute_code(sd_full, orig_test, entry_point, canonical_solution=orig_full, timeout=timeout)
    return {
        'step2_pass': r2['success'],
        'step2_err':  r2.get('error', ''),
        'step3_fail': not r3['success'],
        'step3_err':  r3.get('error', ''),
        'valid':      r2['success'] and not r3['success'],
    }

def save_results(records, path=OUT_PATH):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(records, f, indent=2, ensure_ascii=False)

print('Helpers ready.')

Helpers ready.


In [64]:
## Quick debug: test a single task manually — paste LLM output to inspect
def debug_task(real_task_id: str, strategy: str = 'diff_gate', n_calls: int = 1):
    task = next((t for t in LABELING if t['real_task_id'] == real_task_id), None)
    if task is None:
        print(f'Task {real_task_id!r} not found'); return

    entry_pt  = task['entry_point']
    orig_test = task.get('test', '')
    orig_full = build_full_code(task.get('original_prompt_text',''), task.get('canonical_solution_text',''))

    print(f'── {real_task_id}  entry={entry_pt}  is_meas={task.get("is_measurement")} ──')

    for call in range(1, n_calls + 1):
        print(f'\n--- Call {call}/{n_calls} ---')
        raw = _call_ds(REWRITE_SYS, build_rewrite_msg(task, strategy))
        parsed = parse_rewrite(raw)

        print('Gate Change:', parsed['gate_change'])
        print('New Prompt:',  parsed['new_prompt'][:200])
        print('\n[New Code]\n', parsed['new_code'][:600])
        print('\n[New Test]\n', parsed['new_test'][:600])

        if not parsed['new_code'] or not parsed['new_test']:
            print('✗ parse failed'); continue

        sd_full = parsed['new_code']
        val = validate_sd(sd_full, parsed['new_test'], orig_test, entry_pt, orig_full)
        print('\nStep2 pass:', val['step2_pass'], '|', val['step2_err'][:200] if not val['step2_pass'] else '')
        print('Step3 fail:', val['step3_fail'], '|', val['step3_err'][:200] if not val['step3_fail'] else '')

# Uncomment to debug:
# debug_task('qiskitHumanEval/3', strategy='diff_gate', n_calls=1)

## 4. Choose Strategy & Build Task List

In [ ]:
STRATEGY         = 'diff_gate'   # 'diff_gate' | 'diff_sequence'
INTER_TASK_DELAY = 5                # seconds between tasks

batch_tasks = [t for t in LABELING if t.get('programmer_facing') == 'Yes']

if Path(OUT_PATH).exists():
    _cur = json.load(open(OUT_PATH, encoding='utf-8'))
    _s = {s: sum(1 for r in _cur if r.get('strategy') == STRATEGY and r['status'] == s)
          for s in ('pass', 'test_fixable', 'struct_ok', 'llm_ok', 'reject')}
    print(f'Strategy: {STRATEGY}  |  batch: {len(batch_tasks)}')
    print(f'  pass={_s["pass"]}  test_fixable={_s["test_fixable"]}  '
          f'struct_ok={_s["struct_ok"]}  llm_ok={_s["llm_ok"]}  reject={_s["reject"]}')
else:
    print(f'Strategy: {STRATEGY}  |  batch: {len(batch_tasks)}  (no results yet)')

## 5. Generation Pipeline (3 phases)

| Cell | Role | Blocking point |
|---|---|---|
| **gen_loop** | precheck(orig) → LLM rewrite → parse → `llm_ok` | LLM API call |
| **struct_loop** | check_gate_structure(orig + rewrite) → `struct_ok` | circuit exec |
| **val_loop** | judge (quality gate) → validate_sd (mechanical check) | LLM + execute_test ×2 |

Status flow: `llm_ok` → `struct_ok` → `pass` or `test_fixable`

val_loop order (new):
1. **Judge first** — if REJECT → `reject`
2. **validate_sd** — step2: new_test must PASS on sd_code; step3: orig_test must FAIL on sd_code
   - Both pass → `pass`
   - Either fails → `test_fixable` (code is judge-approved; LLM-generated test is broken)

`test_fixable` records are excluded from gen_loop retries (added to gen_skip). Fix tests manually post-hoc.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Phase 1 — gen_loop: precheck(orig) + LLM + parse → llm_ok / reject
# Skip: pass / test_fixable / struct_ok / llm_ok  |  Retry: reject
# Atomic _upsert re-reads file before each write to avoid concurrent-write clobber.
# ═══════════════════════════════════════════════════════════════════════════════
from qmri_utils import deserialize_kwargs, extract_args_recorder

results = json.load(open(OUT_PATH, encoding='utf-8')) if Path(OUT_PATH).exists() else []

def _upsert(rec):
    current = json.load(open(OUT_PATH, encoding='utf-8')) if Path(OUT_PATH).exists() else []
    idx = next((i for i, r in enumerate(current) if r['rewrite_id'] == rec['rewrite_id']), None)
    if idx is not None:
        current[idx] = rec
    else:
        current.append(rec)
    save_results(current)

def _serial_has_repr(v) -> bool:
    """True if kwargs_serial value contains any repr-typed entry (unrecoverable string)."""
    if isinstance(v, dict):
        if v.get('type') == 'repr':
            return True
        return any(_serial_has_repr(x) for x in v.values())
    if isinstance(v, list):
        return any(_serial_has_repr(x) for x in v)
    return False

gen_skip = {'pass', 'test_fixable', 'struct_ok', 'llm_ok'}
gen_done = {r['rewrite_id'] for r in results
            if r.get('strategy') == STRATEGY and r['status'] in gen_skip}
gen_pending = [t for t in batch_tasks
               if f"{t['real_task_id']}-{STRATEGY}" not in gen_done]

print(f'gen_loop  strategy={STRATEGY}  pending={len(gen_pending)}')

_decomp_pat = re.compile(r'(?i)(design|implement|construct)\s+a\s+\w+\s+gate\s+using')

for i, task in enumerate(gen_pending):
    tid        = task['real_task_id']
    rewrite_id = f'{tid}-{STRATEGY}'
    entry_pt   = task['entry_point']
    orig_full  = build_full_code(task.get('original_prompt_text', ''),
                                 task.get('canonical_solution_text', ''))
    _exc   = EXC_LOOKUP.get(tid, {})
    _ks    = _exc.get('kwargs_serial') or {}
    if isinstance(_ks, str):
        _ks = json.loads(_ks)
    kwargs = deserialize_kwargs(_ks)

    # repr fallback: re-capture real kwargs from test when kwargs_serial has repr-typed values
    if _exc.get('exc_status') == 'ok' and any(_serial_has_repr(v) for v in _ks.values()):
        _test_src = _exc.get('test', '')
        if _test_src:
            _rec_kw, _rec_st, _ = extract_args_recorder(orig_full, _test_src, entry_pt)
            if _rec_st == 'ok' and _rec_kw:
                kwargs = _rec_kw

    print(f'[{i+1}/{len(gen_pending)}] {rewrite_id}', end='  ', flush=True)

    base_rec = {
        'rewrite_id': rewrite_id, 'real_task_id': tid, 'strategy': STRATEGY,
        'source': task.get('source'), 'entry_point': entry_pt,
        'is_measurement': task.get('is_measurement'),
        'is_parameterized': task.get('is_parameterized'),
        'original_prompt_text':    task.get('original_prompt_text', ''),
        'canonical_solution_text': task.get('canonical_solution_text', ''),
    }
    _empty = {'judge_recommendation': None, 'judge_reason': None,
              'rewritten_prompt': '', 'rewritten_code': '', 'test': '', 'gate_change': '',
              'val_step2_pass': None, 'val_step3_fail': None,
              'val_step2_err': '', 'val_step3_err': '', 'llm_raw': None}

    def _reject(reason, attempts=0, raw=None):
        print(f'✗ {reason}')
        _upsert({**base_rec, **_empty, 'status': 'reject',
                 'attempts': attempts, 'reject_reason': reason, 'llm_raw': raw})

    # ── Pre-check (original circuit only, before LLM call) ────────────────────
    orig_ops_pre = _get_gate_ops(orig_full, entry_pt, kwargs)
    if orig_ops_pre is None:
        _reject('struct_precheck: original circuit could not be executed')
        continue
    if len(orig_ops_pre) == 0:
        _reject('struct_precheck: original circuit has no standard named gates')
        continue
    if STRATEGY == 'diff_gate':
        non_std = [g for g, _ in orig_ops_pre if g not in STANDARD_GATES]
        if non_std:
            _reject(f'struct_precheck: original uses non-standard gate(s): {non_std[:3]}')
            continue
    if STRATEGY == 'diff_sequence' and len(orig_ops_pre) < 2:
        _reject('struct_precheck: fewer than 2 gates (nothing to swap)')
        continue
    if STRATEGY == 'diff_sequence' and _decomp_pat.search(task.get('original_prompt_text', '')):
        _reject('task_precheck: gate decomposition task (unsuitable for diff_sequence)')
        continue

    # ── LLM rewrite call (up to 2 retries on empty response) ─────────────────
    raw_rewrite  = None
    empty_misses = 0
    api_err      = False
    for _ in range(3):
        try:
            raw_rewrite = _call_ds(REWRITE_SYS, build_rewrite_msg(task, STRATEGY))
        except Exception as e:
            _reject(f'rewrite_api_error: {e}', attempts=1)
            api_err = True
            break
        if raw_rewrite:
            break
        empty_misses += 1
        print(f'  empty({empty_misses}/2)...', end=' ', flush=True)
        time.sleep(3)

    if api_err:
        continue
    if not raw_rewrite:
        _reject(f'llm_empty: returned empty after {empty_misses} tries', attempts=1, raw='')
        continue

    # ── Parse ─────────────────────────────────────────────────────────────────
    parsed = parse_rewrite(raw_rewrite)
    if not parsed['new_code'] or not parsed['new_test'] or not parsed['new_prompt']:
        _reject(f'parse_error: code={bool(parsed["new_code"])} '
                f'test={bool(parsed["new_test"])} prompt={bool(parsed["new_prompt"])}  '
                f'raw={repr(raw_rewrite[:80])}', attempts=1, raw=raw_rewrite)
        continue

    # ── llm_ok ────────────────────────────────────────────────────────────────
    print('→ llm_ok')
    _upsert({**base_rec, 'status': 'llm_ok', 'attempts': 1, 'reject_reason': None,
             'judge_recommendation': None, 'judge_reason': None,
             'rewritten_prompt': parsed['new_prompt'],
             'rewritten_code':   parsed['new_code'],
             'test':             parsed['new_test'],
             'gate_change':      parsed['gate_change'],
             'val_step2_pass': None, 'val_step3_fail': None,
             'val_step2_err': '', 'val_step3_err': '',
             'llm_raw': raw_rewrite})

    if i < len(gen_pending) - 1:
        time.sleep(INTER_TASK_DELAY)

_final = json.load(open(OUT_PATH, encoding='utf-8'))
n_llm_ok = sum(1 for r in _final if r.get('strategy') == STRATEGY and r['status'] == 'llm_ok')
print(f'\ngen_loop done.  llm_ok={n_llm_ok}')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Phase 3 — val_loop: judge (quality gate) → validate_sd (mechanical check)
# Processes struct_ok records (run struct_loop first to promote llm_ok → struct_ok).
# Order: judge first → if ACCEPT → validate_sd
#   ACCEPT + step2_pass + step3_fail → 'pass'
#   ACCEPT + validate_sd fails      → 'test_fixable' (code approved; auto-test broken)
#   judge REJECT                    → 'reject' (no validate_sd call)
# Atomic _upsert re-reads file before each write.
# ═══════════════════════════════════════════════════════════════════════════════
Val_STRATEGY = 'diff_gate'   # 'diff_gate' | 'diff_sequence'

def _upsert(rec):
    current = json.load(open(OUT_PATH, encoding='utf-8')) if Path(OUT_PATH).exists() else []
    idx = next((i for i, r in enumerate(current) if r['rewrite_id'] == rec['rewrite_id']), None)
    if idx is not None:
        current[idx] = rec
    else:
        current.append(rec)
    save_results(current)

task_map = {t['real_task_id']: t for t in batch_tasks}

_all = json.load(open(OUT_PATH, encoding='utf-8')) if Path(OUT_PATH).exists() else []
val_pending = [r for r in _all
               if r.get('strategy') == Val_STRATEGY and r['status'] == 'struct_ok']

print(f'val_loop  strategy={Val_STRATEGY}  pending={len(val_pending)}')

for i, rec in enumerate(val_pending):
    tid      = rec['real_task_id']
    entry_pt = rec['entry_point']
    print(f'[{i+1}/{len(val_pending)}] {rec["rewrite_id"]}', end='  ', flush=True)

    orig_full = build_full_code(rec['original_prompt_text'], rec['canonical_solution_text'])
    sd_full   = rec['rewritten_code']
    orig_test = task_map.get(tid, {}).get('test', '')

    # ── Step 1: Judge (quality gate — called first, before any code execution) ─
    try:
        raw_judge = _call_judge(JUDGE_SYS, build_judge_msg(
            task_map.get(tid, {}),
            {'new_prompt': rec['rewritten_prompt'], 'new_code': sd_full,
             'gate_change': rec.get('gate_change', '')}
        ))
    except Exception as e:
        print(f'✗ judge_api_error: {e}')
        _upsert({**rec, 'status': 'reject',
                 'reject_reason': f'judge_api_error: {e}',
                 'judge_recommendation': None, 'judge_reason': None,
                 'val_step2_pass': None, 'val_step3_fail': None,
                 'val_step2_err': '', 'val_step3_err': ''})
        continue

    judge_out = parse_judge(raw_judge)
    if judge_out['recommendation'] != 'ACCEPT':
        print(f'✗ judge: {judge_out["reason"][:150]}')
        _upsert({**rec, 'status': 'reject',
                 'reject_reason': f'judge_reject: {judge_out["reason"]}',
                 'judge_recommendation': judge_out['recommendation'],
                 'judge_reason': judge_out['reason'],
                 'val_step2_pass': None, 'val_step3_fail': None,
                 'val_step2_err': '', 'val_step3_err': ''})
        continue

    # ── Step 2+3: validate_sd (only reached when judge says ACCEPT) ───────────
    val = validate_sd(sd_full, rec['test'], orig_test, entry_pt, orig_full)

    if val['step2_pass'] and val['step3_fail']:
        print('✓ pass')
        _upsert({**rec, 'status': 'pass', 'reject_reason': None,
                 'judge_recommendation': 'ACCEPT', 'judge_reason': judge_out['reason'],
                 'val_step2_pass': True, 'val_step3_fail': True,
                 'val_step2_err': '', 'val_step3_err': ''})
    else:
        # Judge approved but auto-generated test is broken → save for manual fixing
        _parts = []
        if not val['step2_pass']:
            _parts.append(f'step2_err: {val["step2_err"][:160]}')
        if not val['step3_fail']:
            _parts.append('step3_still_passes')
        print(f'→ test_fixable  {" | ".join(_parts)[:120]}')
        _upsert({**rec, 'status': 'test_fixable', 'reject_reason': None,
                 'judge_recommendation': 'ACCEPT', 'judge_reason': judge_out['reason'],
                 'val_step2_pass': val['step2_pass'], 'val_step3_fail': val['step3_fail'],
                 'val_step2_err': val['step2_err'], 'val_step3_err': val['step3_err']})

    time.sleep(1)

_final = json.load(open(OUT_PATH, encoding='utf-8'))
n_pass = sum(1 for r in _final if r.get('strategy') == Val_STRATEGY and r['status'] == 'pass')
n_fix  = sum(1 for r in _final if r.get('strategy') == Val_STRATEGY and r['status'] == 'test_fixable')
print(f'\nval_loop done.  pass={n_pass}  test_fixable={n_fix}')

## 6. Stats & Failure Analysis

In [79]:
from collections import Counter

results = json.load(open(OUT_PATH, encoding='utf-8'))

for strat in sorted({r['strategy'] for r in results}):
    sub = [r for r in results if r['strategy'] == strat]
    n_pass   = sum(1 for r in sub if r['status'] == 'pass')
    n_reject = sum(1 for r in sub if r['status'] == 'reject')

    print(f'\n── strategy={strat}  total={len(sub)}  pass={n_pass}  reject={n_reject}')
    print(f'   pass rate: {n_pass/len(sub)*100:.1f}%')

    # Rejection breakdown
    reject_cats = Counter()
    for r in sub:
        if r['status'] == 'reject' and r.get('reject_reason'):
            cat = r['reject_reason'].split(':')[0]
            reject_cats[cat] += 1
    if reject_cats:
        print('   Rejection reasons:')
        for cat, cnt in reject_cats.most_common():
            print(f'     {cat}: {cnt}')

    # Step 2/3 failure detail
    s2_fail = [r for r in sub if r.get('val_step2_pass') is False]
    s3_fail = [r for r in sub if r.get('val_step3_fail') is False]
    print(f'   step2 failed (new_test not passing): {len(s2_fail)}')
    print(f'   step3 failed (orig_test still passes = not diff): {len(s3_fail)}')


── strategy=diff_gate  total=142  pass=32  reject=104
   pass rate: 22.5%
   Rejection reasons:
     struct_precheck: 75
     judge_reject: 12
     validation_step2_fail: 9
     rewrite_api_error: 6
     validation_step3_fail: 2
   step2 failed (new_test not passing): 9
   step3 failed (orig_test still passes = not diff): 2

── strategy=diff_sequence  total=142  pass=8  reject=134
   pass rate: 5.6%
   Rejection reasons:
     struct_precheck: 71
     rewrite_api_error: 24
     validation_step2_fail: 18
     judge_reject: 12
     validation_step3_fail: 5
     task_precheck: 4
   step2 failed (new_test not passing): 18
   step3 failed (orig_test still passes = not diff): 5
